# EDP SCADA · H7 и H30 в Google Colab

Выберите CPU runtime. Скопируйте исходники проекта в Google Drive и откройте этот файл через **File → Open notebook → Google Drive** в Colab. Для приведённых ниже настроек путь к проекту — `MyDrive/scada-pdm-mvp`.

Публичный ZIP Mendeley Version 1 содержит два изображения журнала. В проекте есть их сверенная CSV-транскрипция. Добавьте годовые SCADA `signals_2016.xlsx` и `signals_2017.xlsx` в `data/edp/mendeley_v1/` внутри проекта. Источники — годовые страницы EDP. Если у вас CSV или прямая ссылка на файл, укажите её в `configs/edp_mendeley_v1.json`. Изображения не нужны для выполнения. Подробности в `README.md`.

Запустите ячейки 1–5 по порядку. Первый запуск проверяет покрытие и разбиение без обучения. После сверки журнала укажите `failure_review`, `log_coverage` и при наличии `outages` в конфигурации. Создайте новый кеш, проверьте, какие горизонты получили `eligible`, затем включите обучение в ячейке 1. Ячейки 6–7 покажут метрики, gain и TreeSHAP сохранённых моделей.


In [ ]:
# 1. Настройки и подключение Google Drive
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

USE_DRIVE = True
PROJECT_NAME = "scada-pdm-mvp"
CONFIG_NAME = "edp_mendeley_v1.json"
EXISTING_CACHE = ""  # Абсолютный путь к готовому кешу. Пустая строка создаёт новый.
RUN_TRAINING = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive")
else:
    root = Path("/content")

PROJECT = root / PROJECT_NAME
CONFIG = PROJECT / "configs" / CONFIG_NAME
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SESSION = root / "scada-pdm-runs" / RUN_ID
CACHE = Path(EXISTING_CACHE) if EXISTING_CACHE else SESSION / "cache"
SOURCE_REPORT = SESSION / "sources.json"
OUTPUT = SESSION / ("training" if RUN_TRAINING else "audit")
PYTHON = Path("/content/scada-pdm-venv/bin/python")

if not CONFIG.is_file():
    raise FileNotFoundError(f"Укажите каталог проекта и конфигурацию: {CONFIG}")
if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13), (3, 14)}:
    raise RuntimeError("Для зафиксированных зависимостей требуется Python 3.11–3.14")


In [ ]:
# 2. Установка CPU-зависимостей
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(PYTHON.parent.parent)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(PROJECT / "requirements-ml.txt")], check=True)

def cli(*args, check=True):
    command = [str(PYTHON), "-m", "scada_pdm.cli", *map(str, args)]
    return subprocess.run(command, cwd=PROJECT, check=check)


In [ ]:
# 3. Проверка локальных входов и скачивание по настроенным прямым URL
if EXISTING_CACHE:
    if not (CACHE / "coverage.json").is_file():
        raise FileNotFoundError(f"Отсутствует отчёт готового кеша: {CACHE / 'coverage.json'}")
    print("Будет использован готовый кеш", CACHE)
else:
    status = cli("download", "--config", CONFIG, "--output", SOURCE_REPORT, check=False)
    if not SOURCE_REPORT.is_file():
        status.check_returncode()
    sources = json.loads(SOURCE_REPORT.read_text(encoding="utf-8"))
    missing = [entry["path"] for entry in sources if entry["status"] != "available"]
    if missing:
        raise FileNotFoundError("Загрузите эти файлы в проект на Google Drive: " + ", ".join(missing))
    status.check_returncode()
    print("Список входных файлов и SHA-256", SOURCE_REPORT)


In [ ]:
# 4. Подготовка Parquet, проверка разрывов и разметка H7/H30
if not EXISTING_CACHE:
    cli("prepare", "--config", CONFIG, "--cache", CACHE)
coverage = json.loads((CACHE / "coverage.json").read_text(encoding="utf-8"))
print("Покрытие с доступной историей признаков")
print(json.dumps(coverage["coverage_with_feature_history"], ensure_ascii=False, indent=2))
print("Очередь сверки журнала", CACHE / "failure_review_queue.csv")
print("Каталог кеша", CACHE)


In [ ]:
# 5. Проверка разбиения или обучение Logistic Regression и LightGBM
flags = [] if RUN_TRAINING else ["--audit-only"]
cli("train", "--config", CONFIG, "--cache", CACHE, "--output", OUTPUT, *flags)
report = json.loads((OUTPUT / "metrics.json").read_text(encoding="utf-8"))
for horizon, state in report["horizons"].items():
    print(horizon.upper(), state["status"])
    for fold, counts in state["coverage"].items():
        print(fold, "pos =", counts["positive"], "neg =", counts["negative"],
              "эпизодов =", len(counts["positive_episode_ids"]))
    if state.get("reasons"):
        print("\n".join(state["reasons"]))
print("Каталог отчёта", OUTPUT)


In [ ]:
# 6. Метрики и кривые PR/ROC
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image

metrics_file = OUTPUT / "metrics.csv"
if metrics_file.is_file():
    display(pd.read_csv(metrics_file))
for horizon, state in report["horizons"].items():
    if state["status"] != "trained":
        display(pd.DataFrame.from_dict(state["coverage"], orient="index"))
        continue
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    for model in state["models"]:
        curves = pd.read_csv(OUTPUT / horizon / model / "test_curves.csv")
        for ax, kind, labels in zip(axes, ("PR", "ROC"), (("Recall", "Precision"), ("FPR", "TPR"))):
            values = curves[curves["curve"] == kind]
            ax.plot(values["x"], values["y"], label=model)
            ax.set(xlabel=labels[0], ylabel=labels[1], title=f"{horizon.upper()} · test {kind}")
            ax.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT / f"{horizon}_test_curves.png", dpi=140)
    plt.show()


In [ ]:
# 7. Gain, global/local TreeSHAP и история датчиков
for horizon, state in report["horizons"].items():
    if state["status"] != "trained":
        continue
    outputs = OUTPUT / horizon / "lightgbm"
    print(horizon.upper())
    display(pd.read_csv(outputs / "sensor_importance.csv").head(10))
    for plot in ("gain.png", "shap_global.png", "shap_local.png"):
        display(Image(filename=str(outputs / plot)))
    print("История риск-скора и датчиков:", outputs / "risk_sensor_history.parquet")
print("Артефакты запуска:", OUTPUT)

if not any(state["status"] == "trained" for state in report["horizons"].values()):
    print("Для этого запуска сохранён отчёт покрытия. Файлы SHAP появятся у обученных горизонтов.")
